In [1]:
import numpy as np
import pandas as pd
import random
from collections import deque
from rdflib import Graph
import os

In [2]:
chunk_size = 1

kg_path = r"C:\Users\Sudheera\Documents\Phd\LLMs_and_KGs\clutrr\knowledge_graphs"
percentage_ref_entities = 0.5

In [3]:
def batcher(iterable, n):
    for i in range(0, len(iterable), n):
        yield iterable[i:i + n]


def get_all_entities(graph):
    """
    Extract all entities from the RDF graph.
    """
    entities = set()
    for s, p, o in graph:
        entities.add(s)
        entities.add(o)
    return list(entities)

In [4]:
def build_adjacency_dict(rdf_graph):
    adjacency = {}
    for s, p, o in rdf_graph:
        adjacency.setdefault(s, set()).add(o)
    return adjacency


def bfs_shortest_path(graph, start, goal):
    if start == goal:
        return 0
    visited = set([start])
    queue = deque([(start, 0)])
    while queue:
        current, dist = queue.popleft()
        for neighbor in graph.get(current, []):
            if neighbor == goal:
                return dist + 1
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, dist + 1))
    return float('inf')


def compute_distance_matrix(entities_index, entities_columns, rdf_graph):
    graph = build_adjacency_dict(rdf_graph)
    matrix = pd.DataFrame(index=entities_index, columns=entities_columns)
    for src in entities_index:
        for tgt in entities_columns:
            matrix.loc[src, tgt] = bfs_shortest_path(graph, src, tgt)
    return matrix

## Graph Distance Estimation

In [5]:
# --- 1. Utility: Read and clean distance matrices ---

def format_distance_matrix(df):
    """Load CSV, force numeric, fill missing with 0, and ensure float type."""
    df = df.apply(pd.to_numeric, errors='coerce')
    df = df.fillna(0)
    return df.astype(float)


# --- 2. Convert ref-ref distances to kernel matrix (as in paper) ---

def distance_to_kernel(D_ref_ref, origin_name=None):
    ref_names = list(D_ref_ref.index)
    if origin_name is None:
        origin_name = ref_names[0]
    D_sq = D_ref_ref.values ** 2
    origin_idx = ref_names.index(origin_name)
    D0 = D_sq[origin_idx, :]
    N = len(ref_names)
    K = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            K[i, j] = D0[i] + D0[j] - D_sq[i, j]
    return K, ref_names, origin_idx


# --- 3. Eigendecomposition ---

def kernel_eigendecomposition(K, n_components=None):
    eigenvalues, eigenvectors = np.linalg.eigh(K)
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    if n_components is not None:
        eigenvalues = eigenvalues[:n_components]
        eigenvectors = eigenvectors[:, :n_components]
    return eigenvalues, eigenvectors


# --- 4. Vectorized kernel computation for all others ---

def vectorized_kappa_X(D_other_ref, D_ref_ref, origin_name=None):
    ref_names = list(D_ref_ref.index)
    other_names = list(D_other_ref.columns)
    N = len(ref_names)
    M = len(other_names)
    if origin_name is None:
        origin_name = ref_names[0]
    origin_idx = ref_names.index(origin_name)

    # Squared distances
    D_ref_sq = D_ref_ref.values.astype(float) ** 2  # (N x N)
    D_other_sq = D_other_ref.values.astype(float) ** 2  # (N x M)

    D0_refs = D_ref_sq[origin_idx, :]  # (N,)
    d0_others = D_other_sq[origin_idx, :]  # (M,)

    # Kappa_X: N x M matrix
    kappa_X = D0_refs[:, None] + d0_others[None, :] - D_other_sq  # (N x M)
    return kappa_X, ref_names, other_names


# --- 5. Vectorized Nystrom embedding for all others ---

def vectorized_nystrom_embedding(kappa_X, eigenvalues, eigenvectors):
    UT_kappa = np.dot(eigenvectors.T, kappa_X)  # (n_components x M)
    lambda_inv_sqrt = np.array([1 / np.sqrt(l) if l > 1e-12 else 0 for l in eigenvalues])
    embedding = lambda_inv_sqrt[:, None] * UT_kappa  # (n_components x M)
    return embedding.T  # (M x n_components)


# --- 6. Compute estimated pairwise distances between others ---

def estimated_other_distances_df(embeddings, other_names):
    embeddings = np.array(embeddings, dtype=float)
    diffs = embeddings[:, np.newaxis, :] - embeddings[np.newaxis, :, :]
    est_D = np.linalg.norm(diffs, axis=2)
    return pd.DataFrame(est_D, index=other_names, columns=other_names)


# --- 7. Full pipeline ---

def estimate_distances_between_others(D_ref_ref, D_other_ref, n_components=None, origin_name=None):
    K, ref_names, origin_idx = distance_to_kernel(D_ref_ref, origin_name)
    eigenvalues, eigenvectors = kernel_eigendecomposition(K, n_components)
    kappa_X, ref_names2, other_names = vectorized_kappa_X(D_other_ref, D_ref_ref, origin_name)
    embeddings = vectorized_nystrom_embedding(kappa_X, eigenvalues, eigenvectors)
    return estimated_other_distances_df(embeddings, other_names)

In [6]:
# Get list of file names in the directory (not including folders)
kg_file_list = [f for f in os.listdir(kg_path) if os.path.isfile(os.path.join(kg_path, f))]

for batch in batcher(kg_file_list, chunk_size):
    print(f"Processing batch: {batch}")
    print(f"Number of files in batch: {len(batch)}")
    for kg_file in batch:
        kg_file_path = os.path.join(kg_path, kg_file)
        print(f"Processing file: {kg_file_path}")

        g = Graph()
        g.parse(kg_file_path)
        all_entities = get_all_entities(g)
        # print(f"Entities in {kg_file}: {all_entities}")
        print(f"Number of entities in {kg_file}: {len(all_entities)}")

        num_of_refs = int(len(all_entities) * percentage_ref_entities)
        print(f"Number of reference entities to select: {num_of_refs}")
        # Select a random sample of entities
        # Randomly shuffle the entities and select the first `num_of_refs` entities
        random.shuffle(all_entities)
        ref_entities = all_entities[:num_of_refs]

        ref_ref_distance_matrix = compute_distance_matrix(ref_entities, ref_entities, g)
        ref_ref_distance_matrix = format_distance_matrix(ref_ref_distance_matrix)
        ref_ref_distance_matrix.to_csv("distance_matrix_" + kg_file + ".csv", index=True, header=True)

        # Calculate the distance between the other entities and the reference entities
        ref_other_distance_matrix = compute_distance_matrix(ref_entities, all_entities[num_of_refs:], g)
        ref_other_distance_matrix = format_distance_matrix(ref_other_distance_matrix)
        ref_other_distance_matrix.to_csv("distance_matrix_other_" + kg_file + ".csv", index=True, header=True)

        # Calculate the estimated distances between the other entities
        estimated_distances = estimate_distances_between_others(ref_ref_distance_matrix, ref_other_distance_matrix)
        estimated_distances.to_csv("estimated_distances_" + kg_file + ".csv", index=True, header=True)
    break

Processing batch: ['clutrr_ontology_b20_q28.rdf']
Number of files in batch: 1
Processing file: C:\Users\Sudheera\Documents\Phd\LLMs_and_KGs\clutrr\knowledge_graphs\clutrr_ontology_b20_q28.rdf
Number of entities in clutrr_ontology_b20_q28.rdf: 10
Number of reference entities to select: 5


In [11]:
def strip_before_hash(df: pd.DataFrame) -> pd.DataFrame:
    """
    Return a copy of df where both the index and column names
    have been cleaned by removing everything before (and including) the first '#'.
    """
    # Helper to strip prefix up to '#'
    def _clean_labels(labels):
        # Convert to string index if not already
        labels = labels.astype(str)
        # Remove everything through the first '#'
        return labels.str.replace(r'^.*#', '', regex=True)

    # Create a copy so we don’t mutate the original
    cleaned = df.copy()

    # Clean index
    cleaned.index = _clean_labels(cleaned.index)
    # Clean columns
    cleaned.columns = _clean_labels(cleaned.columns)

    return cleaned

In [12]:
strip_before_hash(ref_ref_distance_matrix)

,Sharon,Patrice,James,William,Steven
Sharon,0.0,5.0,3.0,1.0,4.0
Patrice,5.0,0.0,2.0,6.0,1.0
James,3.0,2.0,0.0,4.0,1.0
William,1.0,6.0,4.0,0.0,5.0
Steven,4.0,1.0,1.0,5.0,0.0


In [13]:
strip_before_hash(ref_other_distance_matrix)

,Ellen,Willie,Cesar,Eula,Steve
Sharon,1.0,3.0,2.0,2.0,3.0
Patrice,4.0,8.0,3.0,7.0,3.0
James,2.0,6.0,1.0,5.0,1.0
William,2.0,2.0,3.0,1.0,4.0
Steven,3.0,7.0,2.0,6.0,2.0


In [14]:
strip_before_hash(estimated_distances)

,Ellen,Willie,Cesar,Eula,Steve
Ellen,0.000000,5.656854,1.414214,4.242641,2.273834
Willie,5.656854,0.000000,7.071068,1.414214,7.930688
Cesar,1.414214,7.071068,0.000000,5.656854,0.859620
Eula,4.242641,1.414214,5.656854,0.000000,6.516474
Steve,2.273834,7.930688,0.859620,6.516474,0.000000


In [15]:
strip_before_hash(compute_distance_matrix(all_entities[num_of_refs:], all_entities[num_of_refs:], g))

,Ellen,Willie,Cesar,Eula,Steve
Ellen,0,4,1,3,2
Willie,4,0,5,1,6
Cesar,1,5,0,4,1
Eula,3,1,4,0,5
Steve,2,6,1,5,0
